In [1]:
from src.tasks import TASKS

sample = TASKS["count_char"].sample()

# Print all field names and values on the Instance object
print(vars(sample))

{'prompt': 'aaabaa;a', 'correct_trace': 'a1 a2 a3 b3 a4 a5', 'wrong_trace': 'b2 c4 a5 c3 b1 a3', 'gold': '5'}


In [9]:
TASKS

{'word_index': Task(name='word_index', chars='abcdefghijklmnopqrstuvwxyz0123456789 ;:\n', block_size=64, max_new_tokens=35, sample=<function _sample_word_index at 0x762634b44180>, chance_acc=0.15652, ceiling_acc=0.89259, description='report the index of a queried letter; trace enumerates (i, char)', answer_pattern='\\d+', bayes_prob=<function _word_index_bayes at 0x76261fcaefc0>, tokenizer=<src.tasks.CharTokenizer object at 0x762634b52db0>),
 'sort_letters': Task(name='sort_letters', chars='abcdefghijklmnopqrstuvwxyz :>\n', block_size=72, max_new_tokens=60, sample=<function _sample_sort_letters at 0x76261fcaf060>, chance_acc=5e-05, ceiling_acc=1.0, description='alphabetize a word; trace is selection sort (remainder > min)', answer_pattern='[a-z]+', bayes_prob=None, tokenizer=<src.tasks.CharTokenizer object at 0x762634b815e0>),
 'multiply': Task(name='multiply', chars='0123456789 :*+=\n', block_size=64, max_new_tokens=48, sample=<function _sample_multiply at 0x76261fcaf1a0>, chance_acc=

In [10]:
import random
import string
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Instance:
    prompt: str
    correct_trace: str
    wrong_trace: str
    gold: str
def _sample_sort_letters() -> Instance:
    L = random.randint(7, 10)
    letters = random.sample(string.ascii_lowercase, L)
    word = "".join(letters)
    gold = "".join(sorted(letters))

    remaining, steps = list(letters), []
    while len(remaining) > 1:
        chosen = min(remaining)
        steps.append(f"{''.join(remaining)}>{chosen}")
        remaining.remove(chosen)
    correct = " ".join(steps)

    remaining, steps = list(letters), []
    while len(remaining) > 1:
        correct_choice = min(remaining)
        wrong_choices = [x for x in remaining if x != correct_choice]
        chosen = random.choice(wrong_choices)

        steps.append(f"{''.join(remaining)}>{chosen}")
        remaining.remove(chosen)
    wrong = " ".join(steps)

    return Instance(word, correct, wrong, gold)

In [11]:
vars(_sample_sort_letters())

{'prompt': 'fjpgiaq',
 'correct_trace': 'fjpgiaq>a fjpgiq>f jpgiq>g jpiq>i jpq>j pq>p',
 'wrong_trace': 'fjpgiaq>j fpgiaq>q fpgia>p fgia>g fia>i fa>f',
 'gold': 'afgijpq'}

In [9]:
import random
from collections import deque
from typing import Dict, List, Tuple

def _sample_graph_path() -> Instance:
    n = random.randint(4, 6)
    nodes = list(range(n))

    # Random sparse undirected graph
    possible = [(i, j) for i in nodes for j in nodes if i < j]
    random.shuffle(possible)
    n_edges = random.randint(n - 1, min(len(possible), n + 2))
    edges = possible[:n_edges]

    adj = {i: [] for i in nodes}
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    s, t = random.sample(nodes, 2)

    # Prompt construction
    edge_str = " ".join(f"{u}-{v}" for u, v in edges)
    prompt = f"{edge_str};{s}>{t}"

    dist, correct_layers = _bfs_shortest(adj, s, t)
    gold = str(dist)
    correct = " ".join(correct_layers)

    # Fix: Match sequence length and initial anchor (s=0) to prevent heuristic shortcuts
    k = len(correct_layers)
    other_nodes = [node for node in nodes if node != s]

    if k - 1 <= len(other_nodes):
        fake_other_nodes = random.sample(other_nodes, k - 1)
    else:
        fake_other_nodes = random.choices(nodes, k=k - 1)

    fake_nodes = [s] + fake_other_nodes
    fake_dists = [0] + [random.randint(1, n) for _ in range(k - 1)]

    wrong_layers = [f"{node}={d}" for node, d in zip(fake_nodes, fake_dists)]

    # Collision guard: Ensure wrong trace is never identical to correct trace
    if wrong_layers == correct_layers:
        fake_dists[-1] = (fake_dists[-1] + 1) % (n + 1)
        wrong_layers = [f"{node}={d}" for node, d in zip(fake_nodes, fake_dists)]

    wrong = " ".join(wrong_layers)

    return Instance(prompt, correct, wrong, gold)

vars(_sample_graph_path())


{'prompt': '2-3 0-5 0-4 0-3 1-3 3-4;3>0',
 'correct_trace': '3=0 2=1 0=1',
 'wrong_trace': '3=0 1=6 4=2',
 'gold': '1'}